# Lesson 21: Extending the Product

## Adding a New Agent to the Team

In this final lesson, we walk through a real scenario: **adding a 4th agent to the team** — a proofreading agent that checks grammar, readability, and SEO best practices.

We trace through the development workflow, and you **verify the output** using everything you've learned.

This is how you'll work in practice:
1. You describe what you want
2. Claude Code implements it
3. You verify the result using your knowledge

## Step 1: Understand

Before making any change, Claude Code explores the codebase. The prompt you'd give:

> "I want to add a proofreading agent to the team. Before making any changes, explore the codebase and explain: (1) how agents are defined, (2) how they're added to the team, (3) what tool pattern to follow."

Claude Code would read:
- `output/backend/agents/content_writer.py` — to see the agent definition pattern
- `output/backend/agents/team.py` — to see how members are assembled
- `output/backend/tools/storage.py` — to see the tool pattern

Do the same — read the key files yourself:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Read an existing agent to understand the pattern
writer_path = os.path.abspath("../../output/backend/agents/content_writer.py")
with open(writer_path, "r", encoding="utf-8") as f:
    print("=== agents/content_writer.py ===")
    print(f.read())

print("\n")

team_path = os.path.abspath("../../output/backend/agents/team.py")
with open(team_path, "r", encoding="utf-8") as f:
    print("=== agents/team.py ===")
    print(f.read())

## Step 2: Plan

After understanding the codebase, you'd tell Claude Code:

> "Plan adding a proofreading agent. It should read an article, check grammar/readability/SEO, fix issues, and save the updated article. Use Claude Sonnet with structured output for the report."

A good plan would include:
1. **New agent file** `agents/proofreader.py`: with `get_article_content` and `update_article_content` tools + `output_schema` for the proofreading report
2. **Schema** defined in the agent file (same as other agents — no separate schemas file)
3. **Add to team** in `agents/team.py`: add `proofreader` to the members list
4. **Update `__init__.py`**: re-export the new agent

Let's build and verify each piece:

## Step 3: Implementation — Schema

First, the schema for structured output. Following the Pydantic pattern from Lesson 11:

In [ ]:
from pydantic import BaseModel, Field

class ProofreadIssue(BaseModel):
    issue_type: str = Field(description="Type: grammar, readability, seo, factual")
    location: str = Field(description="Which section has the issue")
    description: str = Field(description="What the issue is")
    suggestion: str = Field(description="How to fix it")
    severity: str = Field(default="medium", description="low, medium, or high")

class ProofreadResult(BaseModel):
    corrected_article: str = Field(description="Full article with corrections applied")
    issues_found: list[ProofreadIssue] = Field(description="List of issues found and fixed")
    readability_score: int = Field(description="Readability score 1-10")
    seo_score: int = Field(description="SEO optimization score 1-10")
    word_count: int = Field(description="Word count of corrected article")

# Verify the schema works (no API call)
test = ProofreadResult(
    corrected_article="# Test\n\nCorrected article.",
    issues_found=[
        ProofreadIssue(
            issue_type="grammar", location="Paragraph 2",
            description="Run-on sentence", suggestion="Split into two sentences",
        ),
        ProofreadIssue(
            issue_type="seo", location="Introduction",
            description="Target keyword missing", suggestion="Add keyword to first paragraph",
            severity="high",
        ),
    ],
    readability_score=7, seo_score=8, word_count=150,
)

print("Schema validation passed!")
print(f"Issues: {len(test.issues_found)}")
for issue in test.issues_found:
    print(f"  [{issue.severity.upper()}] {issue.issue_type}: {issue.description}")

## Step 3: Implementation — Agent

Claude Code would create `agents/proofreader.py` following the existing pattern:

In [ ]:
# This is what agents/proofreader.py would look like
proofreader_code = '''
"""Proofreader -- checks grammar, readability, SEO best practices."""

from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.storage import get_article_content, update_article_content

proofreader = Agent(
    name="Proofreader",
    role="Check and fix grammar, readability, and SEO issues in articles",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=[get_article_content, update_article_content],
    instructions=[
        "You proofread SEO articles for grammar, readability, and SEO best practices.",
        "Use get_article_content to read the article.",
        "Check for: grammar errors, awkward phrasing, readability issues,"
        " keyword usage, SEO best practices (title, headings, meta).",
        "Fix all issues directly in the article text.",
        "Use update_article_content to save the corrected article.",
        "Report what you found and fixed.",
        "Never use emojis or icons.",
    ],
    markdown=True,
)
'''

print("Agent definition that Claude Code would generate:")
print("=" * 50)
print(proofreader_code)

## Step 3: Implementation — Add to Team

In `agents/team.py`, add the new member:

```python
from .proofreader import proofreader

members = [content_writer, aio_analyzer, proofreader]  # Added!
if image_finder is not None:
    members.append(image_finder)
```

And add to the team leader's instructions:
```python
"- Proofreader: checking grammar, readability, SEO best practices in articles",
```

That's it. The team leader now knows about the proofreader and will delegate proofreading requests to it.

## Step 4: Verify

This is where YOUR knowledge matters. Verify the implementation:

In [ ]:
checks = [
    ("Lesson 5", "Does the agent have a knowledge cutoff issue?",
     "No -- proofreading only needs the article text, not real-time info. No search tools needed."),
    ("Lesson 6", "Are the instructions well-structured?",
     "Yes -- Role ('proofreader'), Task ('check grammar, SEO'), Tools ('get_article_content, update_article_content')."),
    ("Lesson 7", "Is the model choice correct?",
     "Yes -- Claude Sonnet. Supports tools. Good at text analysis and correction."),
    ("Lesson 9", "Are the tools appropriate?",
     "Yes -- get_article_content to read, update_article_content to save. Same tools Image Finder uses."),
    ("Lesson 17", "Does the team integration work?",
     "Yes -- just add to members list and update leader instructions. TeamMode.tasks handles delegation."),
    ("Lesson 18", "Will storage handle the updates?",
     "Yes -- update_article_content is thread-safe. Word count auto-recalculated."),
    ("Lesson 19", "Does the architecture still make sense?",
     "Yes -- one new file (agents/proofreader.py), one line in team.py. Flat architecture maintained."),
]

print("=== Verification Checklist ===\n")
for module, question, answer in checks:
    print(f"  [{module}]")
    print(f"  Q: {question}")
    print(f"  A: {answer}")
    print()

## Vibecoding the Frontend

The React frontend was also built with Claude Code. Here's how:

**The prompt:**
> "I have a FastAPI backend at localhost:7777 with these endpoints: POST /teams/seo-workspace/runs (SSE streaming chat), GET /api/articles (list), GET /api/articles/{id} (detail), DELETE /api/articles/{id}. Build a React + Vite frontend with: a chat interface that streams responses, an article sidebar that lists articles, and an article viewer that renders Markdown."

**What Claude Code generated:**
- `frontend/package.json` — dependencies (react, react-markdown, vite)
- `frontend/vite.config.js` — dev server with proxy to backend
- `frontend/src/api.js` — fetch wrappers with SSE streaming
- `frontend/src/components/Chat.jsx` — streaming chat UI
- `frontend/src/components/ArticleList.jsx` — sidebar with article cards
- `frontend/src/components/ArticleView.jsx` — full article renderer

**The key insight:** You don't need to know React. You describe the **behavior** you want ("stream responses", "list articles in sidebar", "render Markdown") and Claude Code handles the implementation.

**Iterative refinement:**
1. First pass: basic layout and chat
2. "Add SSE streaming so responses appear word by word"
3. "Add an article sidebar that polls for new articles"
4. "Add a delete button on each article card"

Each iteration builds on the previous one. This is **vibecoding** — describing what you want, verifying, iterating.

## More Extension Ideas

Things you could ask Claude Code to build:

| Extension | Difficulty | Claude Code prompt |
|-----------|-----------|-------------------|
| Change writing tone | Easy | "Update Content Writer instructions to write in a casual, friendly tone" |
| Add keyword density check | Medium | "Add a tool that counts keyword frequency in an article" |
| Add translation agent | Medium | "Add a 4th team member that translates articles to Vietnamese using Claude Sonnet" |
| Add Google Search Console | Hard | "Create a GSC toolkit in tools/ following the DataForSEOSearchTools pattern" |
| Add user authentication | Hard | "Add JWT authentication to serve.py using AgentOS middleware" |

Each follows the same workflow: **Understand -> Plan -> Implement -> Verify**.

## The Development Cycle

```
  Idea       "I want to add proofreading"
    |
    v
  Plan       Claude Code explores, you approve the approach
    |
    v
  Implement  Claude Code writes the code
    |
    v
  Verify     YOU check using your knowledge
    |
    v
  Deploy     Test it, use it, iterate
    |
    +-------> (back to Idea for the next improvement)
```

**You are quality control. Claude Code is the builder. Your understanding is what makes you effective.**

Without the knowledge from this course, you couldn't verify if the model choice is right, if the tools are appropriate, or if the team integration is correct. Now you can.

## You've completed all 21 lessons.

The full journey:

### Module 1: Python Basics (Lessons 1-4)
Variables, lists, dictionaries, functions, packages — the foundation for reading code.

### Module 2: Understanding AI (Lessons 5-7)
How LLMs work, prompts and context, model choices — the "why" behind everything.

### Module 3: Building Agents (Lessons 8-13)
First agent, tools, structured output, chaining, mini pipeline — hands-on agent building.

### Module 4: Building the Product (Lessons 14-17)
Claude Code, Content Writer, Images, Team and batch processing — from agents to product.

### Module 5: The Complete Product (Lessons 18-21)
Local storage, how everything connects, web interface, extending with vibecoding.

---

### What's Next?

1. **Start using the product** — run `python output/backend/serve.py` and create articles
2. **Customize agent instructions** — tweak the writing style, tone, SEO approach
3. **Try Claude Code** — make a small change (e.g., change the writer's tone)
4. **Build your own tools** — add tools your SEO team needs
5. **Teach others** — you now understand enough to help your colleagues

**The best way to learn is to build. Start small, verify carefully, iterate quickly.**

In [ ]:
# Final: explore the full project structure
output_dir = os.path.abspath("../../output/backend")
print("Your production codebase:\n")

for root, dirs, files in os.walk(output_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(output_dir, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = len(f.readlines())
            print(f"{sub_indent}{file} ({lines} lines)")

print("\nYou understand every one of these files.")
print("That knowledge is your superpower when working with Claude Code.")